In [ ]:
# Si no se instaló aún, es necesario instalar qiskit
%pip install qiskit[visualization]

---
## Construyendo Compuertas Controladas en Qiskit

En la teoría, hemos aprendido a construir el operador para cualquier compuerta controlada usando proyectores y producto tensorial. Ahora, veremos cómo Qiskit nos proporciona métodos nativos para aplicar estas compuertas de forma sencilla en nuestros circuitos.

La compuerta controlada más fundamental es la **Controlada-NOT (CNOT o CX)**.

In [ ]:
# Reiniciamos el entorno para empezar desde cero
%reset -f

# Importaciones necesarias para esta sección
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from numpy import sqrt

### La Compuerta CNOT (`.cx()`)

El método `.cx(control_qubit, target_qubit)` aplica una compuerta CNOT.

**Recordatorio de la convención de Qiskit:**
- El qubit 0 (`q_0`) es el cable de **arriba**.
- El qubit 1 (`q_1`) es el cable de **abajo**.

Vamos a construir un circuito con el control en `q_1` y el objetivo en `q_0`, que corresponde al operador CNOT que derivamos en la teoría.

In [ ]:
# Creamos un circuito de 2 qubits
qc_cnot = QuantumCircuit(2)

# Aplicamos la CNOT con control en el qubit 1 y objetivo en el qubit 0
qc_cnot.cx(1, 0)

# Dibujamos el circuito
print("Circuito para CNOT con control en q_1:")
display(qc_cnot.draw('mpl'))

### Otras Compuertas Controladas Importantes

Qiskit tiene métodos nativos para otras compuertas controladas comunes:
- **Controlada-Z:** `.cz(control_qubit, target_qubit)`
- **SWAP:** `.swap(qubit_1, qubit_2)` (es simétrica)
- **Toffoli (CCNOT):** `.ccx(control_qubit_1, control_qubit_2, target_qubit)`

In [ ]:
# Creamos un circuito de 3 qubits para mostrar las compuertas
qc_otras = QuantumCircuit(3)

# Compuerta CZ entre q1 y q0
qc_otras.cz(1, 0)

# .barrier() es una directiva visual. No afecta la ejecución,
# pero ayuda a separar lógicamente las secciones de un circuito.
qc_otras.barrier()

# Compuerta SWAP entre q0 y q2
qc_otras.swap(0, 2)
qc_otras.barrier()

# Compuerta Toffoli con controles en q1, q2 y objetivo en q0
qc_otras.ccx(1, 2, 0)

print("Ejemplos de otras compuertas de múltiples qubits:")
display(qc_otras.draw('mpl'))

## Descomponiendo Compuertas en Qiskit

Una de las ideas más poderosas en la computación cuántica es que las compuertas complejas a menudo se pueden construir a partir de un conjunto más simple de compuertas "nativas". En esta sección, vamos a verificar dos de estas identidades fundamentales usando Qiskit.

**Verificación: La Compuerta SWAP a partir de CNOTs**

En la teoría, vimos que una compuerta SWAP se puede implementar usando tres compuertas CNOT. Vamos a construir ese circuito y a demostrar que su operador unitario total es, de hecho, el de una SWAP.

In [ ]:
# Reiniciamos el entorno para empezar desde cero
%reset -f

# Importaciones necesarias
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

# 1. Creamos el circuito que implementa SWAP con 3 CNOTs
qc_swap_from_cnots = QuantumCircuit(2, name="SWAP con CNOTs")

# 2. Construimos la secuencia
qc_swap_from_cnots.cx(0, 1)
qc_swap_from_cnots.cx(1, 0)
qc_swap_from_cnots.cx(0, 1)

# 3. Dibujamos el primer circuito
print("--- Implementación con 3 CNOTs ---")
display(qc_swap_from_cnots.draw('mpl'))

# 4. Extraemos su operador
op_swap_from_cnots = Operator(qc_swap_from_cnots)
print("Matriz resultante:")
display(op_swap_from_cnots.draw('latex'))

# --------------------------------------------------------------------

# 5. Creamos un circuito de referencia con la compuerta SWAP nativa
qc_swap_native = QuantumCircuit(2, name="SWAP Nativo")
qc_swap_native.swap(0, 1)

# 6. Dibujamos el segundo circuito
print("\n--- Implementación con la Compuerta SWAP Nativa ---")
display(qc_swap_native.draw('mpl'))

# 7. Extraemos su operador
op_swap_native = Operator(qc_swap_native)
print("Matriz resultante:")
display(op_swap_native.draw('latex'))

# --------------------------------------------------------------------

# 8. Comparamos ambos operadores
son_iguales = op_swap_from_cnots == op_swap_native
print(f"\n¿Ambos circuitos implementan el mismo operador unitario? {son_iguales}")

**Verificación: La Identidad del "Sándwich de Hadamard"**

Otra identidad crucial es la que relaciona la CNOT y la CZ: `(I ⊗ H) · CNOT · (I ⊗ H) = CZ`.
Esto significa que si "envolvemos" el qubit objetivo de una CNOT con compuertas Hadamard, la operación resultante es una CZ. Vamos a verificarlo.

In [ ]:
# 1. Creamos el circuito que implementa CZ con la identidad H-CNOT-H
qc_h_sandwich = QuantumCircuit(2, name="H-CNOT-H")

# 2. Construimos la secuencia
# Nota: El qubit de control de la CNOT es q1, el objetivo es q0.
qc_h_sandwich.h(0)
qc_h_sandwich.cx(1, 0)
qc_h_sandwich.h(0)

# 3. Dibujamos el primer circuito
print("--- Implementación con H-CNOT-H ---")
display(qc_h_sandwich.draw('mpl'))

# 4. Extraemos su operador
op_h_sandwich = Operator(qc_h_sandwich)
print("Matriz resultante:")
display(op_h_sandwich.draw('latex'))

# --------------------------------------------------------------------

# 5. Creamos un circuito de referencia con la compuerta CZ nativa
qc_cz_native = QuantumCircuit(2, name="CZ Nativo")
qc_cz_native.cz(1, 0)  # El orden aquí no importa, ya que CZ es simétrica

# 6. Dibujamos el segundo circuito
print("\n--- Implementación con la Compuerta CZ Nativa ---")
display(qc_cz_native.draw('mpl'))

# 7. Extraemos su operador
op_cz_native = Operator(qc_cz_native)
print("Matriz resultante:")
display(op_cz_native.draw('latex'))

# --------------------------------------------------------------------

# 8. Comparamos ambos operadores
son_iguales_cz = op_h_sandwich == op_cz_native
print(f"\n¿El circuito H-CNOT-H es equivalente a una compuerta CZ? {son_iguales_cz}")

## Análisis de Estados Entrelazados (Medición Parcial)

En esta sección, vamos a usar las herramientas de simulación de Qiskit para observar directamente las famosas y extrañas correlaciones del entrelazamiento. Veremos cómo la medición de un solo qubit en un par entrelazado afecta instantáneamente al otro, sin importar la distancia.

### Ejercicio 4 (Correlación Perfecta en la Base Z)

In [ ]:
# Reiniciamos el entorno para empezar desde cero
%reset -f

# Importaciones necesarias
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

In [ ]:
# --- Parte A: Creando el estado de Bell |Φ+> ---
qc_bell = QuantumCircuit(2)
qc_bell.h(1)
qc_bell.cx(1, 0)

display(qc_bell.draw("mpl"))

bell_state = Statevector(qc_bell)

print("Estado de Bell |Φ+> creado:")
display(bell_state.draw('latex'))

# --- Parte B: Medición parcial en un bucle ---
print("\n--- Simulación de Medición Parcial (Midiendo q_1) ---")
for i in range(5):
    # Hacemos una copia para no alterar el estado original en cada iteración
    temp_state = bell_state.copy()
    
    # Medimos el qubit 1 (el de la izquierda/abajo)
    resultado, estado_colapsado = temp_state.measure([1])
    
    print(f"Iteración {i+1}:")
    print(f"  - Resultado de medir q_1: '{resultado}'")
    print(f"  - Estado del sistema colapsó a:")
    display(estado_colapsado.draw('latex'))

In [ ]:
# --- Parte C: Medición completa con sample_counts ---
print("\n--- Simulación de 1000 Mediciones Completas ---")
counts = bell_state.sample_counts(shots=1000)
display(plot_histogram(counts))
print("Observa que solo obtenemos '00' y '11', nunca '01' o '10'. ¡Correlación perfecta!")

## Ejercicios

In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 1 (Roles invertidos): CNOT con control en q_0
#------------------------------------------------------------------
# Instrucciones:
# 1. Crea un QuantumCircuit de 2 qubits.
# 2. Aplica una CNOT con control en el qubit 0 (q_0, arriba) y objetivo en el qubit 1 (q_1, abajo).
# 3. Crea el Statevector para el estado inicial |01>.
# 4. Haz evolucionar el estado |01> a través de tu circuito.
# 5. Muestra el estado final. El resultado debería ser |11>.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 2: Obteniendo las matrices CNOT
#------------------------------------------------------------------
# Instrucciones:
# 1. Crea un circuito `qc_10` para la compuerta CNOT(1, 0) (control en q1, objetivo en q0).
# 2. Crea un segundo circuito `qc_01` para la compuerta CNOT(0, 1) (control en q0, objetivo en q1).
# 3. Para cada circuito, extrae su operador usando `Operator()` y muéstralo en formato 'latex'.
# 4. Compara visualmente las dos matrices. ¿Son iguales?

# --- Importa lo que haga falta y completa tu código a continuación ---
...

In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 3: Verificando la Hermiticidad de CNOT
#------------------------------------------------------------------
# Instrucciones:
# 1. Partiendo de los dos operadores CNOT del ejercicio anterior (`op_10` y `op_01`).
# 2. Para cada operador, calcula su adjunto usando el método `.adjoint()`.
# 3. Compara cada operador con su propio adjunto usando `==` para verificar si son Hermitianos.
# 4. Imprime los resultados booleanos. ¿Son los operadores CNOT Hermitianos?

# --- Importa lo que haga falta y completa tu código a continuación ---
# (Puedes reutilizar el código del ejercicio anterior para crear los operadores)
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 4: La "Fábrica" de Estados de Bell
#------------------------------------------------------------------
# Instrucciones:
# El circuito H-CNOT es una "fábrica" que genera los 4 estados de Bell
# a partir de los 4 estados de la base computacional.
#
# Tu tarea es verificarlo para los 4 casos:
# 1. Crea el circuito base: H en q1, seguido de CNOT(1, 0).
# 2. Crea los cuatro estados iniciales: |00>, |01>, |10>, |11>.
# 3. Para cada estado inicial, hazlo evolucionar a través del circuito y muestra
#    el estado final en formato 'latex'.
# 4. Verifica que los resultados son, en orden: |Φ+>, |Ψ+>, |Φ->, |Ψ->.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 5: Verificando los "fallos" en crear entrelazamiento
#------------------------------------------------------------------
# Instrucciones:
# Vamos a comprobar que poner la Hadamard en el lugar incorrecto no crea entrelazamiento.
#
# Parte A (Circuito B de la teoría):
# 1. Crea un circuito con H en q0, seguido de CNOT(1, 0).
# 2. Haz evolucionar el estado |00> a través de este circuito.
# 3. Comprueba si el estado final es separable (usa el test rápido de separabilidad) 
#
# Parte B (Circuito C de la teoría):
# 4. Crea un circuito con H en ambos qubits, seguido de CNOT(1, 0).
# 5. Repite el proceso: haz evolucionar |00> y comprueba si es separable.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 6 (Desafío): Verificando una identidad de compuertas
#------------------------------------------------------------------
# Instrucciones:
# En algunos sistemas cuánticos, la CNOT nativa podría ser CNOT(0,1), pero
# necesitamos implementar una CNOT(1,0). Una identidad útil es:
# CNOT(1,0) = (H ⊗ H) · CNOT(0,1) · (H ⊗ H)
#
# Tu tarea es verificar esta identidad:
# 1. Crea un circuito `qc_identidad` que implemente la secuencia de la derecha:
#    H en ambos qubits, luego CNOT(0,1), luego H en ambos qubits de nuevo.
# 2. Extrae el `Operator` de este circuito.
# 3. Crea el `Operator` para una CNOT(1,0) simple.
# 4. Compara ambos operadores e imprime si son iguales.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 7 (Simetría de CZ): Verificación práctica
#------------------------------------------------------------------
# Instrucciones:
# 1. Crea un circuito `qc1` con una compuerta CZ(1, 0).
# 2. Crea un segundo circuito `qc2` con una compuerta CZ(0, 1).
# 3. Extrae el Operator de cada circuito (`op1` y `op2`).
# 4. Compara ambos operadores usando `==` e imprime el resultado para
#    demostrar que son idénticos.

# --- Importa lo que haga falta y completa tu código a continuación ---
...

# Reiniciamos el entorno para empezar desde cero
%reset -f



In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 8: Construyendo una CNOT a partir de una CZ
#------------------------------------------------------------------
# Instrucciones:
# Vimos que (I⊗H)·CNOT(1,0)·(I⊗H) = CZ. La relación inversa también es cierta:
# (I⊗H)·CZ·(I⊗H) = CNOT(1,0). Esto significa que CNOT y CZ son fundamentalmente la misma
# interacción, vista desde bases diferentes. Son equivalentes bajo un cambio de base en el 
# qubit objetivo. La "transformación de cambio de base" es la compuerta Hadamard!
#
# Tu tarea es verificar esta segunda identidad:
# 1. Crea un circuito de 2 qubits.
# 2. Aplica una H al qubit 0 (el futuro qubit objetivo).
# 3. Aplica una CZ entre q1 y q0 (como CZ es simétrica, cz(1,0) es igual a cz(0,1)).
# 4. Aplica otra H al qubit 0.
# 5. Dibuja el circuito resultante.
# 6. Extrae el Operator de este circuito y compáralo con el Operator
#    de una compuerta CNOT con control en q1 y objetivo en q0.
# 7. Imprime el resultado de la comparación (debería ser True).

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 9: Obteniendo la matriz de la compuerta Toffoli (CCX)
#------------------------------------------------------------------
# Instrucciones:
# 1. Crea un `QuantumCircuit` con 3 qubits.
# 2. Añade una compuerta Toffoli (`.ccx()`). Usa q1 y q2 como controles y q0 como objetivo.
#    (Recuerda la numeración de Qiskit: q2 es el de abajo, q0 el de arriba).
# 3. Dibuja el circuito.
# 4. Extrae el `Operator` del circuito y muestra su matriz de 8x8 en formato 'latex'.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 10: La Toffoli como una Compuerta AND Reversible
#------------------------------------------------------------------
# Instrucciones:
# La compuerta Toffoli es universal para la computación clásica. Podemos usarla
# para implementar una compuerta AND. La idea es: si q2=A y q1=B, entonces
# después de la compuerta, el estado de q0 será C = A AND B.
#
# Tu tarea es verificar esto para las 4 entradas clásicas posibles para A y B.
# 1. Crea un circuito con una compuerta Toffoli (controles en q2, q1; objetivo en q0).
# 2. Para cada uno de los estados iniciales |000>, |010>, |100>, y |110>:
#    a. Crea el Statevector inicial.
#    b. Hazlo evolucionar a través del circuito.
#    c. Imprime el estado inicial y el final para verificar la lógica.
#
# Solo en el caso |110> el bit objetivo q0 debería cambiar a |1>.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 11: Obteniendo la matriz de la compuerta Fredkin (CSWAP)
#------------------------------------------------------------------
# Instrucciones:
# 1. Crea un `QuantumCircuit` con 3 qubits.
# 2. Añade una compuerta Fredkin (`.cswap()`). Usa q2 como control y
#    los qubits q1 y q0 como los objetivos a intercambiar.
# 3. Dibuja el circuito.
# 4. Extrae el `Operator` del circuito y muestra su matriz de 8x8.

# --- Importa lo que haga falta y completa tu código a continuación ---
...


In [ ]:
# Reiniciamos el entorno de ejecución para empezar desde cero
%reset -f

#------------------------------------------------------------------
# Ejercicio 12: La Fredkin como un SWAP Condicional
#------------------------------------------------------------------
# Instrucciones:
# La compuerta Fredkin (CSWAP) realiza un SWAP en dos qubits objetivo,
# condicionado al estado de un qubit de control.
#
# Tu tarea es verificar esta lógica condicional:
# 1. Crea un circuito con una compuerta Fredkin (control en q2, objetivos q1 y q0).
# 2. Caso A (Control = 0):
#    a. Crea el estado inicial |010> (control en |0>, objetivos en |10>).
#    b. Hazlo evolucionar y muestra el resultado. ¿Hubo un SWAP en q1 y q0?
# 3. Caso B (Control = 1):
#    a. Crea el estado inicial |110> (control en |1>, objetivos en |10>).
#    b. Hazlo evolucionar y muestra el resultado. ¿Hubo un SWAP en q1 y q0?

# --- Importa lo que haga falta y completa tu código a continuación ---
...
